In [1]:
# rfm_feature_engineering Yet to start

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed_online_retail.csv")

In [4]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12


In [6]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [7]:
analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
analysis_date

Timestamp('2011-12-10 12:50:00')

In [8]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency
    'Revenue': 'sum'                                           # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346,326,1,77183.60
12347,2,7,4310.00
12348,75,4,1437.24
12349,19,1,1457.55
12350,310,1,294.40


In [9]:
# Recency score (lower recency = better → reverse scoring)
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])

# Frequency score (higher frequency = better)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 
                         5, labels=[1,2,3,4,5])

# Monetary score (higher money = better)
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

In [10]:
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
CustomerID,,,,,,,
12346,326,1,77183.60,1,1,5,115
12347,2,7,4310.00,5,5,5,555
12348,75,4,1437.24,2,4,4,244
12349,19,1,1457.55,4,1,4,414
12350,310,1,294.40,1,1,2,112


In [11]:
rfm[['R_score','F_score','M_score']] = rfm[['R_score','F_score','M_score']].astype(int)


In [12]:
def segment_customer(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4:
        return 'Champions'
    
    elif row['F_score'] >= 4:
        return 'Loyal Customers'
    
    elif row['R_score'] >= 4 and row['F_score'] <= 2:
        return 'Potential Loyalists'
    
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Lost'
    
    else:
        return 'Need Attention'

In [13]:
rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Segment
CustomerID,,,,,,,,
12346,326,1,77183.60,1,1,5,115,Lost
12347,2,7,4310.00,5,5,5,555,Champions
12348,75,4,1437.24,2,4,4,244,Loyal Customers
12349,19,1,1457.55,4,1,4,414,Potential Loyalists
12350,310,1,294.40,1,1,2,112,Lost


In [14]:
rfm['Segment'].value_counts()

Segment
Champions              1136
Lost                   1068
Need Attention          849
Loyal Customers         598
At Risk                 364
Potential Loyalists     319
Name: count, dtype: int64

In [15]:
rfm.to_csv("../data/rfm_data.csv", index=False)